# FASTQ Preprocessing and Quality Control

This notebook introduces a simple paired-end FASTQ preprocessing workflow using **FastQC**, **MultiQC**, and **Trimmomatic**.

**Author:** Juan Sebastian Malagón Torres

## Learning objectives

- Understand the structure of a FASTQ record.
- Generate FastQC reports.
- Summarize reports with MultiQC.
- Remove adapters and low-quality bases with Trimmomatic.
- Repeat quality control after trimming.

> This notebook is intentionally written with simple Bash commands. The commands assume that the notebook is opened from this directory and that the course project keeps the following stable structure:
>
> ```text
> project/
> ├── data/04. Sequencing methods/
> └── material/04. Sequencing methods/
> ```

## 1. FASTQ and paired-end sequencing

A FASTQ record contains four lines:

1. sequence identifier;
2. nucleotide sequence;
3. separator line, usually `+`;
4. per-base quality string.

In paired-end sequencing, `R1` and `R2` are reads from opposite ends of the same DNA fragment. Their pairing must be preserved during preprocessing.

The Phred quality score is:

```text
Q = -10 log10(P_error)
```

A higher Phred score means a lower probability of an incorrect base call.

## 2. Check the input files

The input files are located two levels above this notebook, inside the project `data` directory. The tilde (`~`) represents the current user's home directory when it is needed in a program path.

In [1]:
!pwd
!ls -lh ../../data/04.Sequencing_methods/

/home/jmalagont/Documents/courses/javeriana/bioinformatics/material/04.Sequencing_methods
total 431M
-rw-r--r-- 1 jmalagont jmalagont 138M Sep  3 16:00 SRR15334628_1.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 158M Sep  3 16:05 SRR15334628_2.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont  66M Sep  3 16:07 SRR8439237_1.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont  71M Sep  3 16:09 SRR8439237_2.fastq.gz


In [2]:
# Inspect the first FASTQ record from one R1 file.
!zcat ../../data/04.Sequencing_methods/SRR15334628_1.fastq.gz | head -4

@SRR15334628.1 1/1
GTATATTGTTTTACCCGCAAAAATGAACTGTTCAATGTTGGCACCTGCTGGCGCGGTGGCATCGGTAACCAGACACAATTTGTCGCCTTTCAGACGATTAGCGTTACGAATGTTGGCGTAATCAACATGCAGGCCATCAGCAATAATACCGCAATAAATGTCAGCTTCGTCGAGGATCGCGCCCGCCAGGCCCGGTTCACGACCGGTAATATACGGCATCGCGTTGTACAGATGGGTGGCAAAGGTAATC
+
3AAABFFFDDCFFGGGGGGGGGHGFHFGHGGHGHCDGHHFDHHBFHFHGHGGFDFG1E?FHHHGAEHGFHEHHGGHHGHHHEFFGHGGGHHHHFHGCGHGHHGGDGGG/?GHGHHFHGGCDFHHGHHFGHF2GCGHHHFHGG0FGBHHHHGGGGGHHHGHFFFDCGFHFGGGGGG-:CCDGGCGFGGAGGGGGGGFFFFFFFAFFF@BFFFFFFFFAABFFFA@-@.;BFF/99/ABDFFFFFFEFFFF/

gzip: stdout: Broken pipe


## 3. Create folders for the results

The original FASTQ files will not be modified. Reports and trimmed files will be saved inside the notebook directory.

In [3]:
!mkdir -p results/raw_fastqc
!mkdir -p results/raw_multiqc
!mkdir -p results/trimmed_reads
!mkdir -p results/trimmed_fastqc
!mkdir -p results/trimmed_multiqc

## 4. Raw-read quality control with FastQC

FastQC evaluates several aspects of sequencing quality:

- **Basic Statistics:** number of reads, read length, and encoding;
- **Per-base Sequence Quality:** quality at each position;
- **Per-sequence Quality Scores:** distribution of read quality;
- **Per-base Sequence Content:** nucleotide composition;
- **Sequence Duplication Levels:** possible PCR enrichment or biological repetition;
- **Overrepresented Sequences:** possible adapters, primers, or contaminants;
- **Adapter Content:** adapter contamination across the reads.

The labels `PASS`, `WARN`, and `FAIL` are screening flags. They must be interpreted according to the library type and the downstream analysis.

In [4]:
!fastqc ../../data/04.Sequencing_methods/*.fastq.gz  -o results/raw_fastqc

application/gzip
application/gzip
application/gzip
application/gzip
Started analysis of SRR15334628_1.fastq.gz
Approx 5% complete for SRR15334628_1.fastq.gz
Approx 10% complete for SRR15334628_1.fastq.gz
Approx 15% complete for SRR15334628_1.fastq.gz
Approx 20% complete for SRR15334628_1.fastq.gz
Approx 25% complete for SRR15334628_1.fastq.gz
Approx 30% complete for SRR15334628_1.fastq.gz
Approx 35% complete for SRR15334628_1.fastq.gz
Approx 40% complete for SRR15334628_1.fastq.gz
Approx 45% complete for SRR15334628_1.fastq.gz
Approx 50% complete for SRR15334628_1.fastq.gz
Approx 55% complete for SRR15334628_1.fastq.gz
Approx 60% complete for SRR15334628_1.fastq.gz
Approx 65% complete for SRR15334628_1.fastq.gz
Approx 70% complete for SRR15334628_1.fastq.gz
Approx 75% complete for SRR15334628_1.fastq.gz
Approx 80% complete for SRR15334628_1.fastq.gz
Approx 85% complete for SRR15334628_1.fastq.gz
Approx 90% complete for SRR15334628_1.fastq.gz
Approx 95% complete for SRR15334628_1.fastq.

In [5]:
!ls -lh results/raw_fastqc

total 3.7M
-rw-r--r-- 1 jmalagont jmalagont 634K Sep  6 00:06 SRR15334628_1_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 337K Sep  6 00:06 SRR15334628_1_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 627K Sep  6 00:06 SRR15334628_2_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 352K Sep  6 00:06 SRR15334628_2_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 592K Sep  6 00:06 SRR8439237_1_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 320K Sep  6 00:06 SRR8439237_1_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 592K Sep  6 00:07 SRR8439237_2_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 320K Sep  6 00:07 SRR8439237_2_fastqc.zip


## 5. Combine the raw reports with MultiQC

MultiQC scans the FastQC output and creates a single report for comparison across samples and read directions.

In [6]:
!multiqc results/raw_fastqc --outdir results/raw_multiqc --filename raw_multiqc_report.html --force


/// ]8;id=13265505;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /home/jmalagont/Documents/courses/javeriana/bioinformatics/material/04.Sequencing_methods/results/raw_fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 8/8  0/8  
            fastqc | Found 4 reports
     write_results | Data        : results/raw_multiqc/raw_multiqc_report_data   (overwritten)
     write_results | Report      : results/raw_multiqc/raw_multiqc_report.html   (overwritten)
           multiqc | MultiQC complete


In [7]:
!ls -lh results/raw_multiqc

total 2.3M
drwxr-xr-x 2 jmalagont jmalagont 4.0K Sep  6 00:07 raw_multiqc_report_data
-rw-r--r-- 1 jmalagont jmalagont 2.3M Sep  6 00:07 raw_multiqc_report.html


### Raw quality-control questions

1. Which modules show warnings or failures?
2. Are the same issues present in R1 and R2?
3. Are the issues shared by both samples?
4. Is there evidence of adapter contamination?
5. Does the quality decrease toward the ends of the reads?

## 6. Trimming with Trimmomatic

Trimmomatic removes technical sequences and low-quality regions. In paired-end mode, it creates four output files for each sample:

- paired R1;
- unpaired R1;
- paired R2;
- unpaired R2.

The parameters used below are:

| Parameter | Meaning |
|---|---|
| `ILLUMINACLIP` | Remove adapter sequences |
| `LEADING:3` | Remove low-quality bases at the beginning |
| `TRAILING:3` | Remove low-quality bases at the end |
| `SLIDINGWINDOW:4:15` | Trim when the average quality in four bases is below Q15 |
| `MINLEN:36` | Discard reads shorter than 36 bases |

The adapter path uses `~`, so it begins in the current user's home directory. If Trimmomatic is installed in another location, change only the adapter path in the command.

In [33]:
code = ''' trimmomatic PE -phred33 ../../data/04.Sequencing_methods/SRR15334628_1.fastq.gz \
                         ../../data/04.Sequencing_methods/SRR15334628_2.fastq.gz \
                         results/trimmed_reads/SRR15334628_1.paired.fastq.gz \
                         results/trimmed_reads/SRR15334628_1.unpaired.fastq.gz \
                         results/trimmed_reads/SRR15334628_2.paired.fastq.gz \
                         results/trimmed_reads/SRR15334628_2.unpaired.fastq.gz \
                         LEADING:3 \
                         TRAILING:3 \
                         SLIDINGWINDOW:4:15 \
                         MINLEN:36
        '''


! $code


TrimmomaticPE: Started with arguments:
 -phred33 ../../data/04.Sequencing_methods/SRR15334628_1.fastq.gz ../../data/04.Sequencing_methods/SRR15334628_2.fastq.gz results/trimmed_reads/SRR15334628_1.paired.fastq.gz results/trimmed_reads/SRR15334628_1.unpaired.fastq.gz results/trimmed_reads/SRR15334628_2.paired.fastq.gz results/trimmed_reads/SRR15334628_2.unpaired.fastq.gz LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Input Read Pairs: 796067 Both Surviving: 782793 (98.33%) Forward Only Surviving: 11484 (1.44%) Reverse Only Surviving: 1417 (0.18%) Dropped: 373 (0.05%)
TrimmomaticPE: Completed successfully


In [34]:
code = ''' trimmomatic PE -phred33 ../../data/04.Sequencing_methods/SRR8439237_1.fastq.gz \
                         ../../data/04.Sequencing_methods/SRR8439237_2.fastq.gz \
                         results/trimmed_reads/SRR8439237_1.paired.fastq.gz \
                         results/trimmed_reads/SRR8439237_1.unpaired.fastq.gz \
                         results/trimmed_reads/SRR8439237_2.paired.fastq.gz \
                         results/trimmed_reads/SRR8439237_2.unpaired.fastq.gz \
                         LEADING:3 \
                         TRAILING:3 \
                         SLIDINGWINDOW:4:15 \
                         MINLEN:36
        '''


! $code

TrimmomaticPE: Started with arguments:
 -phred33 ../../data/04.Sequencing_methods/SRR8439237_1.fastq.gz ../../data/04.Sequencing_methods/SRR8439237_2.fastq.gz results/trimmed_reads/SRR8439237_1.paired.fastq.gz results/trimmed_reads/SRR8439237_1.unpaired.fastq.gz results/trimmed_reads/SRR8439237_2.paired.fastq.gz results/trimmed_reads/SRR8439237_2.unpaired.fastq.gz LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Input Read Pairs: 1844088 Both Surviving: 1813500 (98.34%) Forward Only Surviving: 23228 (1.26%) Reverse Only Surviving: 5872 (0.32%) Dropped: 1488 (0.08%)
TrimmomaticPE: Completed successfully


In [35]:
!ls -lh results/trimmed_reads

total 393M
-rw-r--r-- 1 jmalagont jmalagont 127M Sep  6 00:21 SRR15334628_1.paired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 2.0M Sep  6 00:21 SRR15334628_1.unpaired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 130M Sep  6 00:21 SRR15334628_2.paired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 169K Sep  6 00:21 SRR15334628_2.unpaired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont  64M Sep  6 00:22 SRR8439237_1.paired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 1.6M Sep  6 00:22 SRR8439237_1.unpaired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont  68M Sep  6 00:22 SRR8439237_2.paired.fastq.gz
-rw-r--r-- 1 jmalagont jmalagont 417K Sep  6 00:22 SRR8439237_2.unpaired.fastq.gz


## 7. Quality control after trimming

FastQC is run again on the paired trimmed reads. Expected changes include lower adapter content and improved terminal quality. Some reads may be shorter because low-quality bases were removed.

In [36]:
!fastqc results/trimmed_reads/*.paired.fastq.gz -o results/trimmed_fastqc

application/gzip
application/gzip
Started analysis of SRR15334628_1.paired.fastq.gz
application/gzip
application/gzip
Approx 5% complete for SRR15334628_1.paired.fastq.gz
Approx 10% complete for SRR15334628_1.paired.fastq.gz
Approx 15% complete for SRR15334628_1.paired.fastq.gz
Approx 20% complete for SRR15334628_1.paired.fastq.gz
Approx 25% complete for SRR15334628_1.paired.fastq.gz
Approx 30% complete for SRR15334628_1.paired.fastq.gz
Approx 35% complete for SRR15334628_1.paired.fastq.gz
Approx 40% complete for SRR15334628_1.paired.fastq.gz
Approx 45% complete for SRR15334628_1.paired.fastq.gz
Approx 50% complete for SRR15334628_1.paired.fastq.gz
Approx 55% complete for SRR15334628_1.paired.fastq.gz
Approx 60% complete for SRR15334628_1.paired.fastq.gz
Approx 65% complete for SRR15334628_1.paired.fastq.gz
Approx 70% complete for SRR15334628_1.paired.fastq.gz
Approx 75% complete for SRR15334628_1.paired.fastq.gz
Approx 80% complete for SRR15334628_1.paired.fastq.gz
Approx 85% complete

In [37]:
!multiqc results/trimmed_fastqc --outdir results/trimmed_multiqc --filename trimmed_multiqc_report.html --force


/// ]8;id=10719853;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /home/jmalagont/Documents/courses/javeriana/bioinformatics/material/04.Sequencing_methods/results/trimmed_fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 8/8  0/8  
            fastqc | Found 4 reports
     write_results | Data        : results/trimmed_multiqc/trimmed_multiqc_report_data   (overwritten)
     write_results | Report      : results/trimmed_multiqc/trimmed_multiqc_report.html   (overwritten)
           multiqc | MultiQC complete


In [38]:
!ls -lh results/trimmed_fastqc
!ls -lh results/trimmed_multiqc

total 3.8M
-rw-r--r-- 1 jmalagont jmalagont 620K Sep  6 00:22 SRR15334628_1.paired_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 334K Sep  6 00:22 SRR15334628_1.paired_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 623K Sep  6 00:22 SRR15334628_2.paired_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 356K Sep  6 00:22 SRR15334628_2.paired_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 624K Sep  6 00:22 SRR8439237_1.paired_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 319K Sep  6 00:22 SRR8439237_1.paired_fastqc.zip
-rw-r--r-- 1 jmalagont jmalagont 625K Sep  6 00:22 SRR8439237_2.paired_fastqc.html
-rw-r--r-- 1 jmalagont jmalagont 321K Sep  6 00:22 SRR8439237_2.paired_fastqc.zip
total 2.3M
drwxr-xr-x 2 jmalagont jmalagont 4.0K Sep  6 00:23 trimmed_multiqc_report_data
-rw-r--r-- 1 jmalagont jmalagont 2.3M Sep  6 00:23 trimmed_multiqc_report.html


## 8. Compare the raw and trimmed results

Use the two MultiQC reports to answer:

1. Did adapter content decrease?
2. Did terminal base quality improve?
3. Did read lengths change as expected?
4. How many reads remained paired?
5. Were some reads written to the unpaired files?
6. Are the remaining warnings explained by the library design?

The objective is not necessarily to make every module display `PASS`. The objective is to produce reads that are appropriate for the intended downstream analysis.

## Reproducibility checklist

Record the following for a real analysis:

- input FASTQ files;
- tool versions;
- adapter FASTA file;
- trimming parameters;
- numbers of paired and unpaired reads;
- raw and post-trimming QC reports.